# ハンズオン④ 評価実験 — 手法を定量比較する

**対応セクション**: 2-4「評価実験：手法を定量比較する」  
**推奨所要時間**: 約 45 分  
**必要環境**: CPU のみで可（参照モデルが必要な場合は GPU 推奨）

---

## このノートブックの目標

1. 10 問の評価質問セットに対して「素のLLM」「SFT後」「RAGあり」の3条件で回答を取得する
2. ROUGE スコアで N-gram の重複率を計算する
3. BERTScore で意味的な類似度を計算する
4. LLM-as-judge で GPT ライクなモデルに採点させる
5. 3指標を並べて手法間の差を可視化する

## 0. セットアップ

In [ ]:
# 実行時間: 数秒
import os
import json
import torch
import matplotlib.pyplot as plt
import numpy as np

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
os.environ.setdefault('TRANSFORMERS_CACHE', '/data/shared/hf_cache')

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams['axes.unicode_minus'] = False

print('セットアップ完了')

---

## Step 1: 評価用質問セットと参照回答

In [ ]:
# 実行時間: 数秒
# 評価質問セット（10問）と参照回答
eval_set = [
    {
        'question': '機械学習における過学習とは何ですか？',
        'reference': 'モデルが訓練データに過度に適合し、未知のデータに対して汎化できなくなる現象です。対策にはドロップアウト、正則化、早期終了などがあります。',
    },
    {
        'question': 'LoRA でファインチューニングするとき何が学習されますか？',
        'reference': '元のモデルパラメータは凍結し、低ランク行列 B と A のみが学習されます。差分 ΔW = BA として表現され、全パラメータの 0.1〜0.5% 程度だけを更新します。',
    },
    {
        'question': 'RAG で「チャンクサイズ」が精度に影響する理由は何ですか？',
        'reference': 'チャンクが小さすぎると文脈が途切れ、大きすぎるとノイズが増えます。質問に対して最も関連する情報を含む適切な粒度に分割することが重要です。',
    },
    {
        'question': 'SFT と RAG はどう使い分けますか？',
        'reference': '知識の更新頻度が高い場合は RAG、応答スタイルや形式を固定したい場合は SFT が適しています。実務ではハイブリッド構成も多く使われます。',
    },
    {
        'question': 'Self-Attention で Q・K・V はそれぞれ何を表しますか？',
        'reference': 'Q（Query）は検索したい情報、K（Key）は各トークンの索引、V（Value）は実際の情報を表します。Q と K の内積でどのトークンに注目するかを決め、V で情報を集約します。',
    },
    {
        'question': 'DPO の beta パラメータは何を制御しますか？',
        'reference': '参照モデルからの逸脱量（KL ダイバージェンス）へのペナルティ係数です。大きいほど参照モデルに近い保守的な出力になります。',
    },
    {
        'question': 'BERTScore と ROUGE の主な違いは何ですか？',
        'reference': 'ROUGE は N-gram の表層的な重複を見ます。BERTScore は BERT の埋め込みで意味的な類似度を計算するため、言い換えや同義語を適切に評価できます。',
    },
    {
        'question': '4bit 量子化（QLoRA）のメリットは何ですか？',
        'reference': 'モデルパラメータを 4bit で表現することで VRAM 使用量を大幅に削減できます。LLaMA 3 8B の場合、fp16 の約 16GB から約 5GB に削減できます。',
    },
    {
        'question': '教師ありファインチューニング（SFT）の損失関数は何ですか？',
        'reference': '事前学習と同じクロスエントロピー損失です。ただし SFT では回答トークン部分のみを損失計算の対象にします。',
    },
    {
        'question': 'wandb（Weights & Biases）を使う目的は何ですか？',
        'reference': '学習ログ（損失・精度・勾配ノルム等）をリアルタイムで可視化・追跡するためです。複数の実験条件を比較したり、ハイパーパラメータの影響を分析するのに有効です。',
    },
]

print(f'評価質問数: {len(eval_set)}')
for i, item in enumerate(eval_set[:3]):
    print(f'\n[Q{i+1}] {item["question"]}')
    print(f'  参照: {item["reference"][:80]}...')

---

## Step 2: 3条件の回答を用意

実際のモデル推論は時間がかかるため、ここではデモ用の回答を定義します。  
本番では Step 2 をモデル推論に置き換えてください。

In [ ]:
# 実行時間: 数秒
# デモ用の回答（本番はモデル推論に置き換える）
# 各条件の「差」が分かりやすいよう意図的に品質差をつけています

generated_answers = [
    {  # Q1: 過学習
        'base':  '訓練データを覚えすぎてしまうことです。',
        'sft':   'モデルが訓練データに過度に適合し、テストデータでの性能が下がる現象です。ドロップアウトや正則化で対策できます。',
        'rag':   'モデルが訓練データに過度に適合して汎化できなくなる現象です。ドロップアウト、L1/L2正則化、バッチ正規化、早期終了などの対策があります。',
    },
    {  # Q2: LoRA
        'base':  'モデルの重みを学習します。',
        'sft':   '元のパラメータを凍結し、低ランク行列 B と A だけを学習します。全体の 0.1〜0.5% 程度のパラメータで効率的に適応できます。',
        'rag':   '元のモデルパラメータは凍結し、差分を表す低ランク行列 B と A のみが学習されます。ΔW = BA として表現し、全パラメータの 0.1〜0.5% だけを更新します。',
    },
    {  # Q3: チャンクサイズ
        'base':  'チャンクが大きいと検索が難しくなるからです。',
        'sft':   'チャンクサイズが小さいと文脈が途切れ、大きすぎるとノイズが増えます。最適なサイズはタスクによって異なります。',
        'rag':   'チャンクが小さすぎると文脈が途切れて回答品質が下がり、大きすぎると無関係な情報が混入しやすくなります。256・512・1024 tokenを試して最適値を探すことが推奨されます。',
    },
    {  # Q4: SFT vs RAG
        'base':  'どちらもAIを強化する手法です。',
        'sft':   '知識更新が頻繁な場合は RAG、応答スタイルを固めたい場合は SFT が向いています。',
        'rag':   '知識の更新頻度が高い場合は RAG（DBを更新するだけ）、応答スタイル・形式の統一には SFT が適しています。実務では SFT + RAG のハイブリッドも広く使われます。',
    },
    {  # Q5: Q K V
        'base':  'アテンション機構の要素です。',
        'sft':   'Q は検索クエリ、K は索引、V は実際の値です。Q と K の内積でどのトークンに注目するかを決め、V で情報を集約します。',
        'rag':   'Q（Query）は検索したい情報、K（Key）は各トークンの識別子、V（Value）は実際の情報を表します。Q と K の内積で注目度スコアを計算し、V を加重平均して出力を得ます。',
    },
    {  # Q6: beta
        'base':  '学習率のようなものです。',
        'sft':   '参照モデルからの逸脱にペナルティを与える係数です。大きいほど元のモデルに近い出力になります。',
        'rag':   'KL ダイバージェンスへのペナルティ係数で、参照モデルからの逸脱量を制御します。値が大きいほど参照モデルに近い保守的な出力になり、小さすぎると崩壊リスクがあります。',
    },
    {  # Q7: BERTScore vs ROUGE
        'base':  'BERTScoreの方が新しい手法です。',
        'sft':   'ROUGE は単語の重複を見ます。BERTScore は BERT の埋め込みで意味的な類似度を評価するため、言い換えに対して頑健です。',
        'rag':   'ROUGE は N-gram の表層的な重複率を計算します。BERTScore は BERT の文脈依存埋め込みを使って意味的類似度を計算するため、同義表現や言い換えを適切に評価できます。',
    },
    {  # Q8: QLoRA
        'base':  '軽量化できます。',
        'sft':   '4bit 量子化で VRAM を大幅に削減できます。LLaMA 3 8B では 16GB から 5GB 程度に削減できます。',
        'rag':   'パラメータを 4bit で表現することで VRAM 使用量を削減できます。LLaMA 3 8B の場合 fp16 の約 16GB から約 5GB に削減でき、単一の A100 でファインチューニングが可能になります。',
    },
    {  # Q9: SFT 損失関数
        'base':  'クロスエントロピーです。',
        'sft':   '事前学習と同じクロスエントロピー損失ですが、回答トークン部分のみを計算対象にします。',
        'rag':   'クロスエントロピー損失を使います。SFT では指示部分ではなく回答トークン部分のみを損失計算の対象にする点が事前学習との違いです。',
    },
    {  # Q10: wandb
        'base':  '実験を管理するツールです。',
        'sft':   '学習ログをリアルタイムで可視化するためです。損失曲線や勾配ノルムを監視し、学習の異常を早期に発見できます。',
        'rag':   '学習ログ（損失・精度・勾配ノルム等）をリアルタイムで可視化・追跡するためです。複数の実験条件を比較したりハイパーパラメータの影響を分析するのに有効です。',
    },
]

print(f'回答セット数: {len(generated_answers)} 問 × 3 条件')

---

## Step 3: ROUGE スコアの計算

In [ ]:
# 実行時間: 約30秒
from rouge_score import rouge_scorer

scorer_rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

rouge_results = {'base': [], 'sft': [], 'rag': []}

for i, (item, answer) in enumerate(zip(eval_set, generated_answers)):
    ref = item['reference']
    for condition in ['base', 'sft', 'rag']:
        scores = scorer_rouge.score(ref, answer[condition])
        rouge_results[condition].append(scores['rougeL'].fmeasure)

print('=== ROUGE-L スコア（10問の平均）===')
for condition in ['base', 'sft', 'rag']:
    mean = np.mean(rouge_results[condition])
    print(f'{condition:6s}: {mean:.3f}')

---

## Step 4: BERTScore の計算

In [ ]:
# 実行時間: 約2〜3分
from bert_score import score as bert_score

references = [item['reference'] for item in eval_set]
bert_results = {}

for condition in ['base', 'sft', 'rag']:
    hypotheses = [ans[condition] for ans in generated_answers]
    P, R, F1 = bert_score(
        hypotheses, references,
        lang='ja',
        model_type='bert-base-multilingual-cased',
        verbose=False,
    )
    bert_results[condition] = F1.numpy()

print('=== BERTScore F1（10問の平均）===')
for condition in ['base', 'sft', 'rag']:
    mean = np.mean(bert_results[condition])
    print(f'{condition:6s}: {mean:.3f}')

---

## Step 5: LLM-as-judge

ここでは採点プロンプトを使ったシミュレーションを行います。  
本番では実際の LLM（例: Llama3 または Claude API）を呼び出してください。

In [ ]:
# 実行時間: 数秒（シミュレーション）

def llm_judge_score(question: str, reference: str, generated: str) -> int:
    """
    採点プロンプトのテンプレート（実際は LLM を呼び出す）
    デモ用に回答長と参照との単語重複から簡易スコアを返す
    """
    # デモ用: 実際は以下のプロンプトを LLM に投げる
    judge_prompt = (
        f'あなたは公平な評価者です。質問に対する回答を 1〜5 点で採点してください。\n\n'
        f'【質問】\n{question}\n\n'
        f'【参照回答】\n{reference}\n\n'
        f'【評価対象の回答】\n{generated}\n\n'
        f'スコア（1〜5）:'
    )
    # 簡易スコア（デモ用）: 回答長と参照との共通単語比率から算出
    ref_words = set(reference)
    gen_words = set(generated)
    overlap = len(ref_words & gen_words) / max(len(ref_words), 1)
    length_score = min(len(generated) / max(len(reference), 1), 1.5)
    raw = overlap * 3 + length_score * 2
    return min(5, max(1, round(raw)))


llm_judge_results = {'base': [], 'sft': [], 'rag': []}

for item, answer in zip(eval_set, generated_answers):
    for condition in ['base', 'sft', 'rag']:
        score = llm_judge_score(item['question'], item['reference'], answer[condition])
        llm_judge_results[condition].append(score)

print('=== LLM-as-judge スコア（10問の平均）===')
for condition in ['base', 'sft', 'rag']:
    mean = np.mean(llm_judge_results[condition])
    print(f'{condition:6s}: {mean:.2f} / 5.0')

---

## Step 6: 結果の可視化

In [ ]:
# 実行時間: 数秒
conditions = ['素のLLM', 'SFT後', 'RAGあり']
colors = ['#A8A29E', '#1A6B52', '#0D4A38']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# ROUGE-L
rouge_means = [np.mean(rouge_results[k]) for k in ['base', 'sft', 'rag']]
axes[0].bar(conditions, rouge_means, color=colors, width=0.5)
axes[0].set_title('ROUGE-L スコア', fontsize=12)
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('スコア')
for i, v in enumerate(rouge_means):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# BERTScore
bert_means = [np.mean(bert_results[k]) for k in ['base', 'sft', 'rag']]
axes[1].bar(conditions, bert_means, color=colors, width=0.5)
axes[1].set_title('BERTScore F1', fontsize=12)
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel('スコア')
for i, v in enumerate(bert_means):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# LLM-as-judge
judge_means = [np.mean(llm_judge_results[k]) for k in ['base', 'sft', 'rag']]
axes[2].bar(conditions, judge_means, color=colors, width=0.5)
axes[2].set_title('LLM-as-judge スコア', fontsize=12)
axes[2].set_ylim(0, 5.5)
axes[2].set_ylabel('スコア (1〜5)')
for i, v in enumerate(judge_means):
    axes[2].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=10)

plt.suptitle('手法別評価スコア比較（10問平均）', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('./evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n結果を evaluation_results.png に保存しました')

---

## まとめ

| 指標 | 測定対象 | 使い所 |
|---|---|---|
| ROUGE-L | N-gram の表層的重複 | 一次スクリーニング・高速評価 |
| BERTScore F1 | 意味的類似度 | 言い換えを含む評価 |
| LLM-as-judge | 人間評価との相関 | 最終的な品質判断 |

3指標を組み合わせることで、単一指標のバイアスを打ち消し、より信頼性の高い評価ができます。

**第2章完了！**  
第3章では、第2章で強化したモデルをエージェントの頭脳として組み込みます。